# 09 Build the Bantay Santa Luz dashboard — notebook

It is 30 September 2026. The City Mayor wants **one screen** for the Local Health Board:
how many cases, where, is it going up or down, and what do we do. In this notebook Maya
builds a simple version of that screen with Tala's help, step by step:

1. compute the headline numbers (KPIs);
2. make three Plotly charts: epidemic curve, map, dengue against its threshold;
3. put everything into **one self-contained HTML file** with a small template;
4. run Doc Mon's **publish check** before anything leaves the office.

Everything here is computed from the synthetic Santa Luz data. Santa Luz, its people and its
numbers are invented. The file we write goes to your computer's temporary folder, not into
the book.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 · Load the data

We need four files: the clean, de-identified line list, the barangay populations, the weekly
dengue counts and the barangay map (GeoJSON). No names or addresses are in any of them —
chapter 03 removed those.

In [ ]:
import importlib.util
import json
import re
import tempfile
from pathlib import Path
from string import Template

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA = Path("data/synthetic/ph")
ll = pd.read_csv(DATA / "santa_luz_linelist.csv", parse_dates=["date_onset"])
barangays = pd.read_csv(DATA / "santa_luz_barangays.csv")
weekly = pd.read_csv(DATA / "santa_luz_dengue_weekly.csv")
geojson = json.loads((DATA / "santa_luz_barangays.geojson").read_text(encoding="utf-8"))
AS_OF = "30 September 2026"
AGE_GROUPS = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]

print(len(ll), "cases ·", len(barangays), "barangays ·", len(weekly), "weeks of dengue counts")

## Step 2 · How many? The headline numbers

Maya asks Tala:

> *I have a pandas DataFrame `ll`, one row per case, with columns `case_id`, `disease`
> (`leptospirosis` or `dengue`), `outcome` (`alive` or `died`) and `hospitalised` (0/1).
> Made-up rows: `X-1, dengue, alive, 0` · `X-2, leptospirosis, died, 1` · `X-3, dengue,
> alive, 1`. Write a function `compute_kpis(ll)` that returns cases, deaths, hospitalised and
> the case fatality rate in % (one decimal) for each disease and for the whole city.*

Here is what came back:

In [ ]:
def compute_kpis(ll: pd.DataFrame) -> pd.DataFrame:
    """Cases, deaths, hospitalised and CFR (%) by disease, plus a city total row."""
    kpis = ll.groupby("disease").agg(
        cases=("case_id", "size"),
        deaths=("outcome", lambda s: (s == "died").sum()),
        hospitalised=("hospitalised", "sum"),
    )
    kpis.loc["total"] = kpis.sum()
    kpis["cfr_pct"] = (100 * kpis["deaths"] / kpis["cases"]).round(1)
    return kpis


kpis = compute_kpis(ll)
kpis

**Check it.** Two quick checks: run the function on three rows we can count by hand, and
count the deaths a second, different way. The CFR denominator must be *cases*, and the total
row must add up.

In [ ]:
tiny = pd.DataFrame({"case_id": ["X-1", "X-2", "X-3"],
                     "disease": ["dengue", "leptospirosis", "dengue"],
                     "outcome": ["alive", "died", "alive"],
                     "hospitalised": [0, 1, 1]})
tiny_kpis = compute_kpis(tiny)
assert tiny_kpis.loc["dengue", "cases"] == 2 and tiny_kpis.loc["leptospirosis", "cfr_pct"] == 100
assert kpis.loc["total", "deaths"] == (ll["outcome"] == "died").sum()
assert kpis.loc["total", "cases"] == len(ll)
print("Checks passed.")

## Step 3 · The privacy rule we will use everywhere

A dashboard goes to people outside the CESU. A count of **1 to 4** in a small group (one
barangay, one week, one age group) can point to a neighbour. So, before anything is drawn or
printed, counts from 1 to 4 become **"<5"** and their rates are withheld. Zero stays zero:
it describes no one. City totals for the whole period (the KPIs) are published in full, as in
routine bulletins.

In [ ]:
def suppress(counts: pd.Series | pd.DataFrame, threshold: int = 5) -> pd.Series | pd.DataFrame:
    """Replace counts from 1 to threshold - 1 with '<5' (zero stays 0)."""
    small = (counts > 0) & (counts < threshold)
    return counts.astype(object).mask(small, f"<{threshold}")


suppress(pd.Series([0, 1, 4, 5, 12]))

## Step 4 · When? The epidemic curve

Cases by **week** of onset (ISO weeks), both diseases stacked. Weeks are coarse enough not to
reveal anyone's exact onset date. A week with 1–4 cases of one disease is left blank and
labelled "<5" in the hover text. We mark Typhoon Amihan (18 July, in week 29).

In [ ]:
ll["week"] = ll["date_onset"].dt.isocalendar().week
curve = pd.crosstab(ll["week"], ll["disease"]).reindex(range(28, 40), fill_value=0)
public_curve = curve.mask((curve > 0) & (curve < 5))      # 1-4: not drawn
labels = [f"W{w}" for w in curve.index]

fig_curve = go.Figure()
for disease, colour in [("leptospirosis", "#0038A8"), ("dengue", "#E8590C")]:
    fig_curve.add_bar(x=labels, y=public_curve[disease], name=disease.title(),
                      marker_color=colour, customdata=suppress(curve[disease]),
                      hovertemplate="%{x}: %{customdata} cases<extra></extra>")
fig_curve.add_annotation(x="W29", y=curve.sum(axis=1).max(), text="Typhoon Amihan, 18 Jul",
                         showarrow=True, arrowhead=2, ay=-30)
fig_curve.update_layout(barmode="stack", bargap=0.1, height=380,
                        title="Cases by week of symptom onset, 2026",
                        xaxis_title="ISO week of onset", yaxis_title="Cases")
fig_curve

In [ ]:
# Check: the curve keeps every dated case (2 cases have no onset date)
assert curve.to_numpy().sum() == ll["date_onset"].notna().sum() == len(ll) - 2
print(curve.loc[30:34, "leptospirosis"].tolist(), "<- leptospirosis W30-W34; README: 35, 108, 68, 15, 2")

## Step 5 · Where? A map of dengue attack rates

Counts mislead on a map, because big barangays have more cases. We map the **attack rate**:
cases ÷ population × 10,000. A barangay with 1–4 dengue cases shows "<5" and **no rate**
(the rate and the population together would give the count back).

In [ ]:
counts = (ll.groupby(["barangay_id", "disease"]).size().unstack(fill_value=0)
          .reindex(barangays["barangay_id"], fill_value=0))
table = barangays.set_index("barangay_id")[["barangay", "band", "population"]].join(counts)
table["dengue_rate"] = (10_000 * table["dengue"] / table["population"]).round(1)
table.loc[table["dengue"].between(1, 4), "dengue_rate"] = None   # withhold small-cell rates
table["dengue_shown"] = suppress(table["dengue"])

fig_map = px.choropleth(
    table.reset_index(), geojson=geojson, locations="barangay_id",
    featureidkey="properties.barangay_id", color="dengue_rate",
    color_continuous_scale="Oranges", hover_name="barangay",
    hover_data={"barangay_id": False, "dengue_shown": True, "dengue_rate": True},
    labels={"dengue_rate": "per 10,000", "dengue_shown": "cases"},
    title="Dengue attack rate per 10,000 (grey: fewer than 5 cases, rate withheld)")
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(height=420, margin={"l": 0, "r": 0, "t": 50, "b": 0})
fig_map

In [ ]:
# Check the highest rates against the README: Talaba 43.4, Buhangin 35.5
print(table.sort_values("dengue_rate", ascending=False)[["barangay", "dengue_shown", "dengue_rate"]].head(3))
print("Barangays with a withheld dengue rate:", table["dengue_rate"].isna().sum())

## Step 6 · Going up or down? Dengue against the epidemic threshold

The threshold for each ISO week is the **mean + 2 standard deviations** of that week and its
two neighbours in the normal years 2022–2025 (chapter 06). We draw 2026 from week 22.

In [ ]:
base = weekly[weekly["iso_year"].between(2022, 2025)]


def threshold_for(week: int) -> float:
    """Mean + 2 SD of weeks week-1, week, week+1 in the baseline years."""
    window = [(week - 2) % 52 + 1, week, week % 52 + 1]
    values = base.loc[base["iso_week"].isin(window), "cases"]
    return values.mean() + 2 * values.std()


y2026 = weekly[(weekly["iso_year"] == 2026) & (weekly["iso_week"] >= 22)].copy()
y2026["threshold"] = y2026["iso_week"].map(threshold_for)
above = y2026[y2026["cases"] > y2026["threshold"]]
print(len(above), "weeks above the threshold, first:", above["morbidity_week"].iloc[0])

shown = y2026["cases"].mask(y2026["cases"].between(1, 4))
fig_thr = go.Figure()
fig_thr.add_bar(x=y2026["morbidity_week"], y=shown, name="Dengue cases", marker_color="#E8590C",
                customdata=suppress(y2026["cases"]),
                hovertemplate="%{x}: %{customdata} cases<extra></extra>")
fig_thr.add_scatter(x=y2026["morbidity_week"], y=y2026["threshold"], name="Epidemic threshold",
                    mode="lines", line={"color": "#CE1126", "dash": "dash", "shape": "hvh"})
fig_thr.update_layout(height=360, title="Weekly dengue cases against the epidemic threshold, 2026",
                      yaxis_title="Cases")
fig_thr

In [ ]:
# Check against the README: first above in 2026-W32 with 33 cases vs a threshold of 17.5
w32 = y2026.set_index("morbidity_week").loc["2026-W32"]
assert len(above) == 8 and w32["cases"] == 33 and round(w32["threshold"], 1) == 17.5
print("Threshold checks passed. Note: W39 is incomplete (reporting delay).")

## Step 7 · One self-contained HTML file

Maya asks Tala for "a small HTML template with a title, an *as of* date, KPI tiles, the three
charts and a data-notes section, written as ONE file that works offline". Python's
`string.Template` fills the `$placeholders`. `fig.to_html(full_html=False, ...)` turns each
chart into an HTML fragment; the first one carries the Plotly library itself
(`include_plotlyjs=True`), so the file needs no internet.

In [ ]:
PAGE = Template("""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>$title</title>
<style>
  body { font-family: system-ui, sans-serif; max-width: 1100px; margin: 0 auto; padding: 12px;
         background: #fffbf2; color: #1c1b29; }
  header { background: #0038A8; color: #FCD116; padding: 10px 16px; border-radius: 12px; }
  header p { color: #fff; margin: 0; }
  .kpis { display: flex; flex-wrap: wrap; gap: 10px; margin: 12px 0; }
  .kpi { flex: 1 1 140px; background: #fff; border: 2px solid #eadcc0; border-radius: 12px;
         padding: 8px 12px; }
  .kpi b { display: block; font-size: 28px; color: #0038A8; }
  .notes { font-size: 14px; color: #4d4a5e; }
</style></head><body>
<header><h1>$title</h1><p>As of $as_of · synthetic teaching data</p></header>
<section class="kpis">$kpis</section>
$figures
<section class="notes"><h2>Data notes</h2><ul>$notes</ul></section>
</body></html>""")

tiles = [("Cases", kpis.loc["total", "cases"]),
         ("Deaths", kpis.loc["total", "deaths"]),
         ("CFR leptospirosis", f'{kpis.loc["leptospirosis", "cfr_pct"]}%'),
         ("CFR dengue", f'{kpis.loc["dengue", "cfr_pct"]}%'),
         ("Weeks above dengue threshold", len(above))]
notes = ["Santa Luz City and all its data are synthetic, made for teaching.",
         "Aggregated counts only: no names, addresses, case IDs or exact dates of cases.",
         "Counts of 1-4 in any barangay or week are shown as <5 and their rates withheld.",
         "Cases by week of symptom onset; the last week is incomplete (reporting delay)."]

figures = [fig.to_html(full_html=False, include_plotlyjs=(i == 0), config={"responsive": True})
           for i, fig in enumerate([fig_curve, fig_map, fig_thr])]
page_html = PAGE.substitute(
    title="Bantay Santa Luz",
    as_of=AS_OF,
    kpis="".join(f'<div class="kpi">{label}<b>{value}</b></div>' for label, value in tiles),
    figures="\n".join(figures),
    notes="".join(f"<li>{note}</li>" for note in notes),
)
out = Path(tempfile.gettempdir()) / "bantay_santa_luz_simple.html"
out.write_text(page_html, encoding="utf-8")
print(f"Wrote {out} ({out.stat().st_size / 1e6:.1f} MB). Open it in a browser, then on a phone.")

## Step 8 · Doc Mon's publish check

Before the file leaves the CESU, Doc Mon runs through a checklist. Some items can be checked
by code; turn them into a function so you can run it on every version:

- aggregate only — no case IDs, no line-list columns;
- small cells suppressed — no table cell with 1–4;
- an *as of* date and data notes that say what the numbers are.

(Some items stay human: no names or exact addresses in the text, honest axes, tested on a
phone.)

In [ ]:
def publish_check(page: str) -> list[str]:
    """Return the problems Doc Mon would block (an empty list means OK to publish)."""
    problems = []
    if re.search(r"SLC-\d{4}", page):
        problems.append("case IDs found: line-level data must not be published")
    for column in ["case_id", "date_onset", "date_report"]:
        if column in page:
            problems.append(f"line-list column {column!r} found")
    small = re.findall(r"<td>\s*[1-4]\s*</td>", page)
    if small:
        problems.append(f"{len(small)} table cell(s) with 1-4 cases: suppress them")
    for must in ["As of", "Data notes", "synthetic"]:
        if must not in page:
            problems.append(f"missing: {must!r}")
    return problems


print(publish_check(page_html) or "PASS: ready for Doc Mon's final human check")

### Tala got it wrong

The Mayor's staff asked for "more detail", so Maya asked Tala to *add a table of cases by
barangay and age group, and a button to download the data*. Tala's code looked tidy:

```python
detail = pd.crosstab(ll["barangay"], ll["age_group"])          # raw counts, 1s and 2s included
download = f"<script>const DATA = {ll.to_json(orient='records')};</script>"  # the whole line list!
bad_page = page_html.replace("</body>", detail.to_html() + download + "</body>")
```

Let us run it — **in memory only**, never saved — and give it to the publish check.

In [ ]:
detail = pd.crosstab(ll["barangay"], ll["age_group"])
download = f"<script>const DATA = {ll.to_json(orient='records', date_format='iso')};</script>"
bad_page = page_html.replace("</body>", detail.to_html() + download + "</body>")
for problem in publish_check(bad_page):
    print("BLOCKED:", problem)

Two serious problems. Several table cells say **1** or **2**: in a barangay of a few thousand
people, "one case aged 65+" is a person the neighbours can name, not a statistic. And the "download" put
**every row of the line list** — case IDs, ages, exact onset dates — inside the page, where
anyone can read it with *View source*. The fix: publish only aggregates, suppressed.

In [ ]:
safe_detail = suppress(pd.crosstab(ll["band"], ll["age_group"])[AGE_GROUPS])  # coarser, then suppress
fixed_page = page_html.replace("</body>", safe_detail.to_html() + "</body>")
print(publish_check(fixed_page) or "PASS")
safe_detail

## Step 9 · The full site dashboard is the same idea, taken further

The [Bantay Santa Luz dashboard](https://ancientsky.github.io/python4epi/ph/dashboard/) on this
book's site is built by `scripts/build_ph_dashboard.py` with the same steps: `load_data()`,
`compute_kpis()`, `suppress_small_cells()`, `render_html()`. It draws its charts with a small
hand-written script instead of Plotly, so the file stays under 0.2 MB. Its
`suppress_small_cells()` adds one more rule: when a column total is published and only one
cell of that column is hidden, a second cell is hidden too (`x`), so nobody can get the small
one back by subtraction.

The script is part of the repository (the Colab cell cloned it), so we can import it:

In [ ]:
spec = importlib.util.spec_from_file_location("build_ph_dashboard", "scripts/build_ph_dashboard.py")
dashboard = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dashboard)

full = dashboard.compute_kpis(dashboard.load_data())
assert full["cases"]["total"] == kpis.loc["total", "cases"]
assert full["cfr_pct"]["leptospirosis"] == kpis.loc["leptospirosis", "cfr_pct"]
print("Same KPIs as ours:", full["cases"], full["cfr_pct"])

dengue = ll[ll["disease"] == "dengue"]
dashboard.suppress_small_cells(pd.crosstab(dengue["age_group"], dengue["sex"]).reindex(AGE_GROUPS))

In [ ]:
# Build the full dashboard into your temporary folder (it takes about a second)
page = dashboard.build_dashboard("en", Path(tempfile.gettempdir()) / "bantay_full")
print(page, f"{page.stat().st_size / 1e3:.0f} kB")
print(publish_check(page.read_text(encoding="utf-8")) or "PASS")

## Your turn

1. **Add a tile.** Put the number of hospitalised cases into `tiles`, rebuild `page_html` and
   run the publish check again. *Hint:* `kpis.loc["total", "hospitalised"]`.
2. **Map the other disease.** Make `fig_map` show leptospirosis attack rates. *Hint:* copy the
   three `dengue_…` lines in Step 5 and change `dengue` to `leptospirosis`. Which band lights
   up?
3. **A stricter rule.** Some offices suppress counts under 10. How many barangays lose their
   dengue rate with `threshold=10`? *Hint:* `(suppress(table["dengue"], 10) == "<10").sum()`.

In [ ]:
# Your turn 1: add a hospitalised tile

In [ ]:
# Your turn 2: map leptospirosis

In [ ]:
# Your turn 3: a stricter suppression rule

## Before you send it

Open the HTML file **on a phone** (send it to yourself, not to a group chat), read every
number aloud once, and ask Doc Mon — a person — for the final yes. The code catches the
mechanical mistakes; people catch the rest.